In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, balanced_accuracy_score, accuracy_score
from sklearn.model_selection import train_test_split

from E import MeanRegressor
from F import MostFrequentClassifier
from G import CityMeanRegressor
from H import RubricCityMedianClassifier

In [2]:
pd.set_option("display.max_columns", None)

data = pd.read_csv("organisations.csv")
features = pd.read_csv("features.csv")
rubrics = pd.read_csv("rubrics.csv")

features_dict = features.set_index("feature_id").to_dict()["feature_name"]
rubric_dict = rubrics.set_index("rubric_id").to_dict()["rubric_name"]

In [3]:
data

,org_id,city,average_bill,rating,rubrics_id,features_id
0,15903868628669802651,msk,1500.0,4.270968,30776 30774,3501685156 3501779478 20422 3502045016 3502045...
1,16076540698036998306,msk,500.0,4.375000,30771,1509 1082283206 273469383 10462 11617 35017794...
2,8129364761615040323,msk,500.0,4.000000,31495,10462 11177 11617 11629 1416 1018 11704 11867 ...
3,15262729117594253452,msk,500.0,4.538813,30776 30770,3501618484 2020795524 11629 11617 1018 11704 2...
4,13418544315327784420,msk,500.0,4.409091,31495,11617 10462 11177 1416 11867 3501744275 20282 ...
...,...,...,...,...,...,...
68334,4379286080707082909,msk,NaN,3.812500,30774,1018 1415 10462 11629 11867 20422 20424 118949...
68335,7916477189329738565,msk,NaN,4.894231,30776,11634 11629 3501481353 11177 3501773763 11867 ...
68336,12358902585434046825,msk,NaN,4.156250,30774,20422 11867 246 3501754799 3501779478 12048 35...
68337,1712093598996183140,spb,NaN,NaN,30771 30774,3491142672 3501481353 11867 20422 273469383 11...


In [4]:
features

,feature_id,feature_name
0,1,prepress_and_post_printing_processing
1,40,products
2,54,printing_method
3,77,fuel
4,79,shop
...,...,...
1001,3502053162,rating_of_moscow_schools_enum
1002,3502060253,michelin
1003,3502063466,card_big_button
1004,3502071401,hotel_city_center_distance_meters


In [5]:
rubrics

,rubric_id,rubric_name
0,30519,"Булочная, пекарня"
1,30770,"Бар, паб"
2,30771,Быстрое питание
3,30774,Кафе
4,30775,Пиццерия
5,30776,Ресторан
6,30777,Столовая
7,31286,Спортбар
8,31350,Кондитерская
9,31375,Суши-бар


In [6]:
data = data[data.average_bill <= 2500]
print(f"Answer to task 3: {data.average_bill.size}")

cafe_data = data[data.rubrics_id.str.contains("30774")]

mean_msk = cafe_data[cafe_data.city == "msk"].average_bill.mean()
mean_spb = cafe_data[cafe_data.city == "spb"].average_bill.mean()
print(f"Answer to task 4: {round(mean_msk - mean_spb)}")

Answer to task 3: 32136
Answer to task 4: 142


In [7]:
mean_r = data[data.rubrics_id.str.contains("30776")].average_bill.mean()
mean_p = data[data.rubrics_id.str.contains("30770")].average_bill.mean()

train_data, test_data = train_test_split(
    data, stratify=data.average_bill, test_size=0.33, random_state=42
)

# Source model is in E.py
reg = MeanRegressor()
# reg.fit(y=train_data["average_bill"])
# print(reg.predict(test_data['average_bill']))

clf = MostFrequentClassifier()


# clf.fit(y=train_data["average_bill"])
# print(clf.predict(test_data["average_bill"]))


In [8]:
def check_model(model, train_data, test_data, mode):
    model.fit(X=train_data.loc[:, train_data.columns != "average_bill"], y=train_data["average_bill"])
    y_pred = model.predict(test_data.loc[:, test_data.columns != "average_bill"])
    y_true = test_data["average_bill"]
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    rmse = float(rmse)
    if mode == "r":
        return type(model), rmse
    bas = balanced_accuracy_score(y_true, y_pred)
    bas = float(bas)
    acs = accuracy_score(y_true, y_pred)
    return type(model), rmse, bas, acs

In [9]:
print(check_model(reg, train_data, test_data, "r"))
print(check_model(clf, train_data, test_data, "c"))

(<class 'E.MeanRegressor'>, 448.7143889551622)
(<class 'F.MostFrequentClassifier'>, 514.7517402382093, 0.2, 0.6947666195190948)


In [10]:
city_reg = CityMeanRegressor()
# city_reg.fit(X=train_data, y=train_data["average_bill"])

print(check_model(city_reg, train_data, test_data, "r"))

(<class 'G.CityMeanRegressor'>, 445.1063281403263)


In [11]:
rubrics_id_combinations = train_data.rubrics_id.value_counts()

def filter_rubrics(row):
    rubrics_id = row["rubrics_id"]
    new_val = 'other'
    if rubrics_id in rubrics_id_combinations and rubrics_id_combinations[rubrics_id] >= 100:
        new_val = rubrics_id
    row["modified_rubrics"] = new_val
    return row


data['modified_rubrics'] = 'other'
data = data.apply(filter_rubrics, axis=1)
train_data, test_data = train_test_split(
    data, stratify=data.average_bill, test_size=0.33, random_state=42
)

C:\Users\vav11\AppData\Local\Temp\ipykernel_8568\3745526105.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['modified_rubrics'] = 'other'


In [12]:
r_clf = RubricCityMedianClassifier()
print(check_model(r_clf, train_data, test_data, "c"))

(<class 'H.RubricCityMedianClassifier'>, 393.96675836287915, 0.30552511833185647, 0.7095709570957096)


In [13]:
data

,org_id,city,average_bill,rating,rubrics_id,features_id,modified_rubrics
0,15903868628669802651,msk,1500.0,4.270968,30776 30774,3501685156 3501779478 20422 3502045016 3502045...,30776 30774
1,16076540698036998306,msk,500.0,4.375000,30771,1509 1082283206 273469383 10462 11617 35017794...,30771
2,8129364761615040323,msk,500.0,4.000000,31495,10462 11177 11617 11629 1416 1018 11704 11867 ...,31495
3,15262729117594253452,msk,500.0,4.538813,30776 30770,3501618484 2020795524 11629 11617 1018 11704 2...,30776 30770
4,13418544315327784420,msk,500.0,4.409091,31495,11617 10462 11177 1416 11867 3501744275 20282 ...,31495
...,...,...,...,...,...,...,...
68328,17662684569129497551,spb,1000.0,4.561707,30776,246 3501779478 1018 3501618484 3501481353 3501...,30776
68329,5700899951016592061,msk,1500.0,4.377129,31401,3501773763 10462 1018 246 3501779478 350175479...,31401
68330,4686040819909966338,msk,1500.0,3.666667,30776,10462 1189498238 11629 1416 1415 11741 3501481...,30776
68331,12499715465202129892,msk,1500.0,4.554577,30770 30776 30774,273469383 10462 21247 1509 1416 3501618484 350...,other


In [14]:
train_data['modified_features'] = train_data.rubrics_id + " q " + train_data.features_id
train_data

,org_id,city,average_bill,rating,rubrics_id,features_id,modified_rubrics,modified_features
45769,3276960721840719260,msk,500.0,4.500000,30770,11704 20422 1018 11177 1416 11867 10462,30770,30770 q 11704 20422 1018 11177 1416 11867 10462
39061,8452997364765928283,msk,1500.0,4.442623,30774 30776,1415 3501481355 1416 11629 10462 1524 20422 11...,30774 30776,30774 30776 q 1415 3501481355 1416 11629 10462...
59281,14240408259222214074,spb,1000.0,4.018868,30776 30774,3502045032 11741 3502045016 10462 11704 350177...,30776 30774,30776 30774 q 3502045032 11741 3502045016 1046...
51225,15114069072602161053,msk,1500.0,4.364742,31401 30776,3501513153 3501779478 3491142672 273469383 350...,other,31401 30776 q 3501513153 3501779478 3491142672...
29587,2730337118800634815,msk,1000.0,4.698718,30770,21247 10896 3491142672 11629 3501481353 350148...,30770,30770 q 21247 10896 3491142672 11629 350148135...
...,...,...,...,...,...,...,...,...
64667,15641319025413596274,msk,500.0,4.510753,30771,20424 3501744275 273469383 10462 11177 11617 1...,30771,30771 q 20424 3501744275 273469383 10462 11177...
47309,2049892259403324519,msk,500.0,4.333333,30771,273469383 20424 11704 11629 10462 20422 1018,30771,30771 q 273469383 20424 11704 11629 10462 2042...
26208,12224074314753892871,msk,500.0,5.000000,30775,21247 11867 11629 1524 1509 20422 1416 1415 10...,30775,30775 q 21247 11867 11629 1524 1509 20422 1416...
48599,16581456988770474074,msk,500.0,4.692308,31495 30774,3491142672 20282 3501637468 11741 3501745827 3...,31495 30774,31495 30774 q 3491142672 20282 3501637468 1174...


In [15]:
test_data['modified_features'] = test_data.rubrics_id + " q " + test_data.features_id
mask = ~test_data['modified_features'].isin(train_data['modified_features'])
test_data.loc[mask, 'modified_features'] = 'other'
test_data

,org_id,city,average_bill,rating,rubrics_id,features_id,modified_rubrics,modified_features
65841,14385912302763770021,spb,1000.0,4.748444,30776 30770 31401,11177 3501618484 10462 3501481355 1509 1416 20...,other,other
48882,16695436192794975203,msk,500.0,3.793758,30771,3501744275 273469383 3501513153 11617 10462 11...,30771,other
33711,11841431940065207518,msk,500.0,3.606557,30771 30777,3501773763 3501744275 3501773764 3501618484 15...,other,other
33544,16028521499441205186,msk,2000.0,4.683841,30776,3501618484 20422 1082283206 11704 11629 21247 ...,30776,other
35293,12477116204055673498,spb,500.0,4.165394,30776 31401 30770,1524 246 11704 1018 3501618484 2020795524 2124...,other,other
...,...,...,...,...,...,...,...,...
55337,9041226080397910513,msk,2500.0,4.408108,30776,11629 11704 10462 11617 3501744275 20424 35017...,30776,other
64048,14998683880343589209,msk,1000.0,3.555556,30776,273469383 20424 20422 246 1416 11867 11629 104...,30776,other
22010,1621254442333414922,msk,2000.0,4.402516,30776,273469383 21247 11867 1082283206 20422 246 101...,30776,other
40089,5620614742257813954,msk,500.0,NaN,30771,11704 1018 273469383 10462 20422,30771,30771 q 11704 1018 273469383 10462 20422


In [16]:
from I import LargeClassifier


l_clf = LargeClassifier()
l_clf.fit(train_data)

In [17]:
ans = l_clf.predict(test_data)
ans.to_csv("I.csv")

In [18]:
check_model(l_clf, train_data, test_data, "c")

C:\Users\vav11\anaconda3\envs\conda310\lib\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


(I.LargeClassifier, 513.9898108867789, 0.2010249213051401, 0.6934464875058934)

In [20]:
train_pred = l_clf.predict(train_data)
train_true = train_data["average_bill"]

test_pred = l_clf.predict(test_data)
test_true = test_data["average_bill"]

train_rmse = np.sqrt(mean_squared_error(train_true, train_pred))
train_bas = balanced_accuracy_score(train_true, train_pred)

test_rmse = np.sqrt(mean_squared_error(test_true, test_pred))
test_bas = balanced_accuracy_score(test_true, test_pred)

train_rmse = round(train_rmse, 2)
train_bas = round(train_bas, 2)
test_rmse = round(test_rmse, 2)
test_bas = round(test_bas, 2)
print(train_rmse, train_bas, test_rmse, test_bas)

32.42 0.99 513.99 0.2


C:\Users\vav11\anaconda3\envs\conda310\lib\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\vav11\anaconda3\envs\conda310\lib\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
